In [1]:
import pandas as pd
df = pd.read_csv("Hydrogen_Uncertainty_dataset.csv")

In [2]:
df

,"Natural Gas Cost ($/MMBtu, HHV)",Absorber CO2 Removal Efficiency (%),Capacity Factor (%),CO2 Captured (tons/hr),Hydrogen Production Cost: Total Cost: Levelized Cost ($/lb H2)
0,7.975000,96.3675,76.5375,83.798065,1.184908
1,10.424999,95.7825,73.5625,83.289360,1.412497
2,7.825000,97.4025,92.6375,84.698059,1.118635
3,3.325000,94.2075,61.4875,81.919785,0.850518
4,8.525001,90.3375,84.0625,78.554565,1.198089
...,...,...,...,...,...
195,2.425000,96.4575,84.5875,83.876320,0.666642
196,7.675000,94.7475,87.5625,82.389351,1.117000
197,4.275000,98.7075,87.0375,85.832848,0.828909
198,2.675000,94.4775,66.9125,82.154579,0.761790


In [3]:
df.isna().sum()

Natural Gas Cost ($/MMBtu, HHV)                                   0
Absorber CO2 Removal Efficiency (%)                               0
Capacity Factor (%)                                               0
CO2 Captured (tons/hr)                                            0
Hydrogen Production Cost: Total Cost: Levelized Cost ($/lb H2)    0
dtype: int64

In [4]:
df.duplicated().sum()

np.int64(0)

In [5]:
x = df[['Natural Gas Cost ($/MMBtu, HHV)',
    'Absorber CO2 Removal Efficiency (%)',
    'Capacity Factor (%)']]

y = df['CO2 Captured (tons/hr)']

In [6]:
x

,"Natural Gas Cost ($/MMBtu, HHV)",Absorber CO2 Removal Efficiency (%),Capacity Factor (%)
0,7.975000,96.3675,76.5375
1,10.424999,95.7825,73.5625
2,7.825000,97.4025,92.6375
3,3.325000,94.2075,61.4875
4,8.525001,90.3375,84.0625
...,...,...,...
195,2.425000,96.4575,84.5875
196,7.675000,94.7475,87.5625
197,4.275000,98.7075,87.0375
198,2.675000,94.4775,66.9125


In [7]:
y

0      83.798065
1      83.289360
2      84.698059
3      81.919785
4      78.554565
         ...    
195    83.876320
196    82.389351
197    85.832848
198    82.154579
199    79.141525
Name: CO2 Captured (tons/hr), Length: 200, dtype: float64

In [8]:
from sklearn.model_selection import GridSearchCV, train_test_split, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
from sklearn.ensemble import GradientBoostingRegressor

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

print("\n=== REGRESSION MODELS ===\n")


=== REGRESSION MODELS ===



In [9]:
# Linear Regression
reg_pipe = Pipeline([
    ('scaler',StandardScaler()),
    ('reg',LinearRegression())
])
reg_grid = GridSearchCV(reg_pipe, {}, cv=5, scoring='r2', n_jobs=-1)
reg_grid.fit(x_train,y_train)
print("Linear Regression best params:",reg_grid.best_params_)

best_reg = reg_grid.best_estimator_
print('Train R2:',best_reg.score(x_train,y_train))
print('Test R2:',best_reg.score(x_test,y_test))

Linear Regression best params: {}
Train R2: 0.9999999999960957
Test R2: 0.9999999999953594


In [10]:
# SVR
svr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", SVR())
])
svr_grid = GridSearchCV(
    svr_pipe,
    {
        "svr__kernel": ["linear", "rbf"],
        "svr__C": [0.01, 0.1, 1, 10,100],
        "svr__gamma": ["scale"]
    },
    cv=5, scoring="r2", n_jobs=-1
)
svr_grid.fit(x_train, y_train)

best_svr = svr_grid.best_estimator_
print("SVR best params:", svr_grid.best_params_)
print('Train R2:',best_svr.score(x_train,y_train))
print('Test R2:',best_svr.score(x_test,y_test))

SVR best params: {'svr__C': 1, 'svr__gamma': 'scale', 'svr__kernel': 'linear'}
Train R2: 0.999317842365241
Test R2: 0.9993138324430433


In [11]:
# XGBoost Regressor
xgb_grid = GridSearchCV(
    XGBRegressor(random_state=42),
    {
        "n_estimators": [100, 300],
        "max_depth": [3, 6, 10],
        "learning_rate": [0.01, 0.1, 0.3],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0]
    },
    cv=5, scoring="r2", n_jobs=-1
)
xgb_grid.fit(x_train, y_train)

best_xgb = xgb_grid.best_estimator_
print("XGBoost best params:", xgb_grid.best_params_)
print('Train R2:',best_xgb.score(x_train,y_train))
print('Test R2:',best_xgb.score(x_test,y_test))

XGBoost best params: {'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 100, 'subsample': 0.8}
Train R2: 0.9999958941023509
Test R2: 0.9991947686654947


In [12]:
# Gradient Boosting Regressor
gbr_grid = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    {
        "n_estimators": [100, 300],
        "max_depth": [3, 6, 10],
        "learning_rate": [0.01, 0.1, 0.3],
        "subsample": [0.8, 1.0],
        "max_features": [0.8, 1.0]
    },
    cv=5, scoring="r2", n_jobs=-1
)
gbr_grid.fit(x_train, y_train)

best_gbr = gbr_grid.best_estimator_
print("Gradient Boosting best params:", gbr_grid.best_params_)
print('Train R2:', best_gbr.score(x_train, y_train))
print('Test R2:', best_gbr.score(x_test, y_test))

Gradient Boosting best params: {'learning_rate': 0.1, 'max_depth': 3, 'max_features': 1.0, 'n_estimators': 100, 'subsample': 0.8}
Train R2: 0.9999802487487541
Test R2: 0.999650090707374


In [13]:
# Random Forest Regressor
rfr_grid = GridSearchCV(
    RandomForestRegressor(random_state=42),
    {
        "n_estimators": [100,300],
        "max_depth": [None, 5, 10, 20],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ['sqrt',0.5, 0.7]
    },
    cv=5, scoring="r2", n_jobs=-1
)
rfr_grid.fit(x_train, y_train)

best_rfr = rfr_grid.best_estimator_
print("RF best params:", rfr_grid.best_params_)
print('Train R2:',best_rfr.score(x_train,y_train))
print('Test R2:',best_rfr.score(x_test,y_test))

RF best params: {'max_depth': None, 'max_features': 0.7, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}
Train R2: 0.9992146813097798
Test R2: 0.9931747737977149


In [14]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge

poly_pipe = Pipeline([
    ('poly',   PolynomialFeatures(include_bias=False)),
    ('scaler', StandardScaler()),
    ('reg',    Ridge())
])
poly_grid = GridSearchCV(
    poly_pipe,
    {'poly__degree': [2, 3, 4],
     'reg__alpha':  [0.001, 0.01, 0.1, 1, 10, 100]},
    cv=5, scoring='r2', n_jobs=-1
)
poly_grid.fit(x_train, y_train)

best_poly = poly_grid.best_estimator_
print("Polynomial best params:", poly_grid.best_params_)
print('Train R2:', best_poly.score(x_train, y_train))
print('Test R2:', best_poly.score(x_test, y_test))

Polynomial best params: {'poly__degree': 4, 'reg__alpha': 0.001}
Train R2: 0.9999999598642122
Test R2: 0.9999999558171385


In [15]:
#Checking the RMSE and MAPE scores
from sklearn.metrics import mean_squared_error
import numpy as np

print("\n=== RMSE & MAPE ===\n")

models = {
    'Linear Regression': best_reg,
    'SVR':               best_svr,
    'Random Forest':     best_rfr,
    'XGBoost':           best_xgb,
    'Gradient Boost':    best_gbr,
    'Polynomial':        best_poly
}

for name, model in models.items():
    y_pred = model.predict(x_test)
    rmse   = np.sqrt(mean_squared_error(y_test, y_pred))
    mape   = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
    print(f"{name} | RMSE: {rmse:.4f} | MAPE: {mape:.4f}%")


=== RMSE & MAPE ===

Linear Regression | RMSE: 0.0000 | MAPE: 0.0000%
SVR | RMSE: 0.0594 | MAPE: 0.0655%
Random Forest | RMSE: 0.1875 | MAPE: 0.1471%
XGBoost | RMSE: 0.0644 | MAPE: 0.0687%
Gradient Boost | RMSE: 0.0424 | MAPE: 0.0457%
Polynomial | RMSE: 0.0005 | MAPE: 0.0005%


In [16]:
#Exporting to excel
import os
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

file       = 'Hydrogen_CO2_Captured.xlsx'
experiment = 'Target_1_Hydrogen_CO2_Captured'   # change per experiment

results = []

models = {
    'Linear Regression': best_reg,
    'SVR':               best_svr,
    'Random Forest':     best_rfr,
    'XGBoost':           best_xgb,
    'Gradient Boost':    best_gbr,
    'Polynomial':        best_poly
}

for name, model in models.items():
    y_pred = model.predict(x_test)
    results.append({
        'Experiment': experiment,
        'Model':      name,
        'Train R2':   round(model.score(x_train, y_train), 4),
        'Test R2':    round(model.score(x_test, y_test), 4),
        'MAE':        round(mean_absolute_error(y_test, y_pred), 4),
        'MSE':        round(mean_squared_error(y_test, y_pred), 4),
        'RMSE':       round(np.sqrt(mean_squared_error(y_test, y_pred)), 4),
        'MAPE (%)':   round(np.mean(np.abs((y_test - y_pred) / y_test)) * 100, 4)
    })

results_df = pd.DataFrame(results)

if os.path.exists(file):
    existing = pd.read_excel(file)
    combined = pd.concat([existing, results_df], ignore_index=True)
    combined.to_excel(file, index=False)
else:
    results_df.to_excel(file, index=False)

print(results_df)

                       Experiment              Model  Train R2  Test R2  \
0  Target_1_Hydrogen_CO2_Captured  Linear Regression    1.0000   1.0000   
1  Target_1_Hydrogen_CO2_Captured                SVR    0.9993   0.9993   
2  Target_1_Hydrogen_CO2_Captured      Random Forest    0.9992   0.9932   
3  Target_1_Hydrogen_CO2_Captured            XGBoost    1.0000   0.9992   
4  Target_1_Hydrogen_CO2_Captured     Gradient Boost    1.0000   0.9997   
5  Target_1_Hydrogen_CO2_Captured         Polynomial    1.0000   1.0000   

      MAE     MSE    RMSE  MAPE (%)  
0  0.0000  0.0000  0.0000    0.0000  
1  0.0540  0.0035  0.0594    0.0655  
2  0.1223  0.0351  0.1875    0.1471  
3  0.0565  0.0041  0.0644    0.0687  
4  0.0377  0.0018  0.0424    0.0457  
5  0.0004  0.0000  0.0005    0.0005  
